<a href="https://colab.research.google.com/github/RobJavVar/DataSciencePsychNeuro/blob/master/ExerciseSubmissions/13_resampling-methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 13:  Resampling methods

This homework assignment is designed to give you practice with bootstrapping and permutation tests.

You will need to download the **unrestricted_trimmed_1_7_2020_10_50_44.csv** file from the *Homework/hcp_data* folder in the class GitHub repository.

This data is a portion of the [Human Connectome Project database](http://www.humanconnectomeproject.org/). It provides measures of cognitive tasks and brain morphology measuresments from 1206 participants. The full description of each variable is provided in the **HCP_S1200_DataDictionary_April_20_2018.csv** file in the *Homework/hcp_data* folder in the class GitHub repository.

---
## 1. Loading & Visualizing the Data (1 point)

Use the `setwd` and `read.csv` functions to load data from the **unrestricted_trimmed_1_7_2020_10_50_44.csv** file.

(a) Using the tidyverse tools, make a new dataframe `d1` that only inclues the subject ID (`Subject`), gender (`Gender`, self reported at time of data collection), Flanker Task performance (`Flanker_Unadj`), total intracranial volume (`FS_IntraCranial_Vol`), total white matter volume (`FS_Tot_WM_Vol`), and total grey matter volume (`FS_Total_GM_Vol`) variables and remove all _na_ values.

Use the `head` function to look at the first few rows of each data frame.

In [ ]:
# setwd("~/Documents/GitHub/DataSciencePsychNeuro/Exercise datasets/hcp_data")
# lex <- read.csv("unrestricted_trimmed_1_7_2020_10_50_44.csv")
# library(tidyverse)
d1 <- lex %>%
    select(Subject, Gender, Flanker_Unadj, FS_IntraCranial_Vol, 
    FS_Tot_WM_Vol, FS_Total_GM_Vol) %>%
    drop_na()
head(d1)
dim(d1)


(b) Plot grey matter volume (x axis) against intracranial volume (y axis) and Gender (point color).

In [ ]:
# library(ggplot2)
ggplot(data = d1, aes(x = FS_Total_GM_Vol, y = FS_IntraCranial_Vol, color = Gender)) + geom_point()


What patterns do you observe in the scatter plot?

There appears to be a linear relationship, with men measuring higher for both variables.

---
## 2. Logistic classifier (2 points)

We want to try predicting gender using the neural data you have loaded.

(a) Run a logisic regression model to predict gender from total white matter volume, total grey matter volume, and intracranial volume.

In [ ]:
d1 <- d1 %>%
    mutate(male = if_else(Gender == "M", 1, 0))
model1 <- glm(male ~ FS_Total_GM_Vol + FS_IntraCranial_Vol + FS_Tot_WM_Vol, data = d1, family = "binomial")
summary(model1)


Which factors are signficantly associated with gender?

> All 3 are significantly associated with gender. Being male shows a significant association is increased measurements in all 3 variables
>

(b) Estimate the prediction accuracy of your model (Note: this is the training set accuracy). Set your prediction threshold to 0.5.

In [ ]:
d1$probs <- predict(model1, type = "response")
d1$pred_male <- if_else(d1$probs >= 0.5, 1, 0)
accuracy <- mean(d1$pred_male == d1$male, na.rm = TRUE)
print(paste0("Training Accuracy: ", round(accuracy * 100, 2), "%"))


What is the prediction accuracy for gender from the full model?

>82.03%
>

---
## 3. Bootstrapped accuracy (3 points)

Use bootstrapping to estimate the confidence intervals of the _prediction accuracy_ of your model (i.e., the confidence of the correlation between $\hat{y}$ and $y$). Plot the histogram of the bootstrapped prediction accuracies and estimate the confidence intervals off of the standard deviation from the bootstrap.


In [ ]:
install.packages("boot")
library(boot)
# The function needs two inputs: Data, Index
boot.fn <- function(data, index){  
    # return: throw this as output
    # coef: extract coefficients from model object 
    return(coef(glm(male ~ FS_Total_GM_Vol + FS_IntraCranial_Vol + FS_Tot_WM_Vol, 
        data = d1, subset = index, family = "binomial")))}
boot_obj = boot(d1 ,boot.fn ,R=1000) #R=repetitions 
print(boot_obj) #t1 is the intercept and t2 is the horsepower coeff.
print(boot.fn(d1, 1:))

In [ ]:
hist(boot(d1 ,boot.fn ,R=1000)$t[,2], xlab="FS_Total_GM_Vol") #we get a distribution of all of the estimates
#Confidence Interval: .00002 +- .000003

In [ ]:
hist(boot(d1 ,boot.fn ,R=1000)$t[,3], xlab="FS_IntraCranial_Vol") #we get a distribution of all of the estimates
#Confidence Interval: .000005 +- .000001

In [ ]:
hist(boot(d1 ,boot.fn ,R=1000)$t[,4], xlab="FS_Tot_WM_Vol") #we get a distribution of all of the estimates
#Confidence Interval: .000002 +- .000003

How robust is the prediction accuracy of the full model?

> It's very robust, we are getting almost the exact same outputs here.
>

---
## 4. Permutation test for grey matter effects (3 points)

Now run a permutation test, with 1000 iterations, to evaluate how much grey matter volume contributes to the prediction accuracy. Compare the prediction accuracy of the full (unpermuted model) with the distribution of accuracies you get with a randomized grey matter volume term using a histogram (Hint: use the `abline` function to show the original accuracy on the histogram).

In [ ]:
# First let's make a copy of the data set that we'll keep permuting
permd1 = d1 #want to preserve the non-permuted, true form of data!

# Set the number of iterations
R=1000

perm_accuracies <- numeric(R) # Vector to store 1000 accuracy scores
permd1 <- d1

for (i in 1:R) {
  # Shuffle only the Gray Matter Volume
  permd1$FS_Total_GM_Vol <- sample(d1$FS_Total_GM_Vol)
  
  # Re-fit the model with the shuffled data
  perm_model <- glm(male ~ FS_Total_GM_Vol + FS_IntraCranial_Vol + FS_Tot_WM_Vol, 
                    data = permd1, family = "binomial")
  
  # Calculate accuracy for this shuffled model
  perm_probs <- predict(perm_model, type = "response")
  perm_pred <- if_else(perm_probs >= 0.5, 1, 0)
  perm_accuracies[i] <- mean(perm_pred == d1$male, na.rm = TRUE)
}


In [ ]:
# 1. First, make sure we remove any potential NAs from the results
perm_accuracies <- perm_accuracies[!is.na(perm_accuracies)]

# 2. Plot the histogram
# We'll let R handle the x-axis limits automatically first to avoid the error
hist(perm_accuracies, 
     main = "Permutation Test: Gray Matter Volume",
     xlab = "Prediction Accuracy", 
     col = "lightblue",
     breaks = 30)+ # More breaks usually looks better for 1000 iterations

# 3. Add the vertical line for your actual model accuracy
# Use the true_accuracy variable we calculated earlier
abline(v = accuracy, col = "red", lwd = 2, lty = 2)+

# 4. Optional: Add a label to the line so it's clear
text(accuracy, 0, labels = "True Accuracy", pos = 4, col = "red", cex = 0.8)


How much does the grey matter volume influence the prediction accuracy of the model?

> A little bit. All of the predicted accuracies are lower than the accuracy we actually observed when GM was included in the model (82%), but th average accuracy now is almost 81%, so it's pretty close.
>

---
## 5. Reflection (1 point)

Differentiate the bootstrap from a permutation test. Describe each and when is it appropriate to each.

> A bootstrap is when you replace certain rows of a dataframe with repeated observations from that dataframe. So row 5 might be in there twice instead of including row 7. 
A permutation test is when you randomize 1 variable by shuffling all the observations for that variable within a dataset. This is for testing if it actually has an impact on the performance of your model. If random observations do just as well, you might as well remove the variable from your model. So that is the case when you would want to use a permutation test.
You use bootstrapping to determine the overall performance and CIs of your predictor variables after running your model. So this is basically to validate if the accuracy you observe on your training data is actually reliable by seeing if it replicates over multiple iterations.
>

**DUE:** 5pm EST, March 24, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> I asked gemini to help me with the syntax since especially with the last one i wasn't sure how to store the accuracies and gneerate the histogram. 